In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd
import matplotlib
import sklearn

print("Library Versions:")
print(f"  numpy     : {np.__version__}")
print(f"  pandas    : {pd.__version__}")
print(f"  matplotlib: {matplotlib.__version__}")
print(f"  sklearn   : {sklearn.__version__}")

Library Versions:
  numpy     : 2.0.2
  pandas    : 2.2.2
  matplotlib: 3.10.0
  sklearn   : 1.6.1


In [3]:
df = pd.read_csv("/content/StudentsPerformance.csv")

print("First 10 rows:")
print(df.head(10))
print(f"\nDataset shape (rows, cols): {df.shape}")

First 10 rows:
   gender race/ethnicity parental level of education         lunch  \
0  female        group B           bachelor's degree      standard   
1  female        group C                some college      standard   
2  female        group B             master's degree      standard   
3    male        group A          associate's degree  free/reduced   
4    male        group C                some college      standard   
5  female        group B          associate's degree      standard   
6  female        group B                some college      standard   
7    male        group B                some college  free/reduced   
8    male        group D                 high school  free/reduced   
9  female        group B                 high school  free/reduced   

  test preparation course  math score  reading score  writing score  
0                    none          72             72             74  
1               completed          69             90             88  
2   

In [4]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

df[["math score", "reading score"]] = imputer.fit_transform(
    df[["math score", "reading score"]]
)

print("Missing values after imputation:")
print(df[["math score", "reading score"]].isnull().sum())

Missing values after imputation:
math score       0
reading score    0
dtype: int64


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

ct = ColumnTransformer(
    transformers=[("ohe", OneHotEncoder(sparse_output=False), ["lunch"])],
    remainder="passthrough"
)

df_encoded = ct.fit_transform(df)

ohe_cols = ct.named_transformers_["ohe"].get_feature_names_out(["lunch"])
other_cols = [c for c in df.columns if c != "lunch"]
df_encoded = pd.DataFrame(df_encoded, columns=list(ohe_cols) + other_cols)

print("After OneHot Encoding 'lunch':")
print(df_encoded.head(3))

After OneHot Encoding 'lunch':
  lunch_free/reduced lunch_standard  gender race/ethnicity  \
0                0.0            1.0  female        group B   
1                0.0            1.0  female        group C   
2                0.0            1.0  female        group B   

  parental level of education test preparation course math score  \
0           bachelor's degree                    none       72.0   
1                some college               completed       69.0   
2             master's degree                    none       90.0   

  reading score writing score  
0          72.0            74  
1          90.0            88  
2          95.0            93  


In [6]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df_encoded["test preparation course"] = le.fit_transform(
    df_encoded["test preparation course"]
)

print("Label-encoded 'test preparation course' (first 5):")
print(df_encoded["test preparation course"].head())
print(f"Classes: {le.classes_} → {list(range(len(le.classes_)))}")

Label-encoded 'test preparation course' (first 5):
0    1
1    0
2    1
3    1
4    1
Name: test preparation course, dtype: int64
Classes: ['completed' 'none'] → [0, 1]


In [7]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
score_cols = ["math score", "reading score", "writing score"]

df_encoded[score_cols] = scaler.fit_transform(df_encoded[score_cols])

print("Normalized score columns (first 5 rows):")
print(df_encoded[score_cols].head())

Normalized score columns (first 5 rows):
   math score  reading score  writing score
0        0.72       0.662651       0.711111
1        0.69       0.879518       0.866667
2        0.90       0.939759       0.922222
3        0.47       0.481928       0.377778
4        0.76       0.734940       0.722222


In [8]:
from sklearn.model_selection import train_test_split

lunch_cols = [c for c in df_encoded.columns if c.startswith("lunch_")]
feature_cols = score_cols + lunch_cols

X = df_encoded[feature_cols]
y = df_encoded["test preparation course"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print(f"X shape      : {X.shape}")
print(f"Training samples : {X_train.shape[0]}")
print(f"Testing  samples : {X_test.shape[0]}")
print("\nPreprocessing complete!")

X shape      : (1000, 5)
Training samples : 750
Testing  samples : 250

Preprocessing complete!
